In [1]:
import pickle   # importing pickle for saving and loading machine learning models
from sklearn.model_selection import train_test_split  # importing train_test_split for spliting the data into training and testing
from imblearn.over_sampling import SMOTE  # importing SMOTE for Balancing the Data
import warnings
warnings.filterwarnings('ignore')

In [2]:
from preprocessing import *

ModuleNotFoundError: No module named 'preprocessing'

In [ ]:
with open('Tree_CT.pkl','rb') as f:
    pre = pickle.load(f)

In [ ]:
pre

In [ ]:
with open('Processed_data.pkl','rb') as f:
    df = pickle.load(f)

In [ ]:
df

In [ ]:
X = df.drop("Attrition",axis=1)     # Extract the features (all columns except Attritions) from the dataset
y = df["Attrition"].map({"No":0,"Yes":1})  # Extract the target variable from the dataset with converting 0 and 1.

In [ ]:
df.Attrition.value_counts()

In [ ]:
# Spliting the data into train and test
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=33)

In [ ]:
# Transform the training data using the preprocessor object or PipeLine
processed_x_train = pre.fit_transform(x_train)

In [ ]:
processed_x_train

In [ ]:
processed_x_train[0]

### Model Building

In [ ]:
import sklearn
sklearn.set_config(print_changed_only=False)

In [ ]:
from sklearn.tree import DecisionTreeClassifier        #importing decision tree from sklearn.tree
dt=DecisionTreeClassifier()             #object creation for decision tree
dt.fit(processed_x_train,y_train)    # sample_weight=sample_weights    #train the model

In [ ]:
#Prediction
processed_x_test = pre.transform(x_test)
y_pred=dt.predict(processed_x_test)

In [ ]:
#Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results = pd.DataFrame(index=['Accuracy','F1'])
results['DT/imb'] = [acc,f1]
results

In [ ]:
#Prediction on Training data - Test of overfitting
y_pred1=dt.predict(processed_x_train)

#Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
acc = accuracy_score(y_train,y_pred1)
pr = precision_score(y_train,y_pred1)
re = recall_score(y_train,y_pred1)
f1 = f1_score(y_train,y_pred1)
cm = confusion_matrix(y_train,y_pred1)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_train,y_pred1))

In [ ]:
results['DT/Train'] = [acc,f1]
results

In [ ]:
y_train.value_counts()

In [ ]:
## Data Balancing and Creating Model
from imblearn.over_sampling import SMOTE
x_smote,y_smote = SMOTE().fit_resample(processed_x_train,y_train)

dt_bal=DecisionTreeClassifier()            
dt_bal.fit(x_smote,y_smote)   #Training the model again with smote balanced data

In [ ]:
y_smote.value_counts()

In [ ]:
#Prediction
#processed_x_test = pre.transform(x_test)
y_pred=dt_bal.predict(processed_x_test)

In [ ]:
#Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['DT/Smote'] = [acc,f1]
results

In [ ]:
from sklearn.model_selection import GridSearchCV
#creating dictionary--> key value pair of hyperparameters having key as parameter and values as its values
params = {
    "criterion":("gini", "entropy"), #Split criterion
    "splitter":("best", "random"),  #searches the features for a split
    "max_depth":(list(range(1,20,2))),  #depth of tree range from 1 to 19
    "min_samples_split":[2,5,7,10,12,15],    #the minimum number of samples required to split internal node
    "min_samples_leaf":list(range(1, 20)), #minimum number of samples required to be at a leaf node,we are passing list which is range from 1 to 19
}


tree_clf = DecisionTreeClassifier()                # object creation for decision tree with random state 3
tree_cv = GridSearchCV(tree_clf, params, scoring="f1", n_jobs=-1, verbose=5, cv=5)

In [ ]:
tree_cv.fit(processed_x_train,y_train)    # training data on gridsearch cv
best_params = tree_cv.best_params_    # it will give you best parameters
print(f"Best paramters: {best_params})")   # printing  best parameters

In [ ]:
##Fitting 5 folds for each of 7524 candidates, totalling 37620 fits ---- takes huge time.

In [ ]:
tree_cv.best_params_    # getting best parameters from cv

In [ ]:
# passing best parameter to decision tree
dt_optimal=DecisionTreeClassifier(criterion='gini',max_depth=7,min_samples_leaf=14,min_samples_split=5,splitter='best')
dt_optimal.fit(processed_x_train,y_train)

y_pred=dt_optimal.predict(processed_x_test)

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['DT/HPT'] = [acc,f1]
results

## Random Forest Implementation

In [ ]:
from sklearn.ensemble import RandomForestClassifier   # importing randomforest

rf_clf = RandomForestClassifier() # Assigning RandomForest CLassifier into variable
rf_clf.fit(processed_x_train,y_train)   # training the data
y_pred=rf_clf.predict(processed_x_test) 

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
rf_clf

In [ ]:
results['RF'] = [acc,f1]
results

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
n_estimators = [int(x) for x in np.linspace(start=200, stop=2000, num=5)]      # List Comprehension-using for loop in list
max_features = ['auto', 'sqrt','log2']        # maximum number of features allowed to try in individual tree
max_depth = [int(x) for x in np.linspace(10, 110, num=5)]            # List Comprehension-using for loop in list
max_depth.append(None)
min_samples_split = [5,7,10,13]          # minimum number of samples required to split an internal node
min_samples_leaf = [2, 4]              # minimum number of samples required to be at a leaf node.

#dictionary for hyperparameters
random_grid = {'n_estimators': n_estimators, 'max_features': max_features,
               'max_depth': max_depth, 'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf}

rf_clf1 = RandomForestClassifier(random_state=42)   # Loading the model

rf_cv = RandomizedSearchCV(estimator=rf_clf1, scoring='f1',param_distributions= random_grid, cv=3,
                               verbose=2, n_jobs=-1)

In [ ]:
np.linspace(200,2000,5)

In [ ]:
rf_cv.fit(processed_x_train, y_train)
rf_best_params = rf_cv.best_params_ 
print(f"Best paramters: {rf_best_params})") 

In [ ]:
# passing best parameter to randomforest
rf_clf2 = RandomForestClassifier(n_estimators=650,min_samples_leaf=4,min_samples_split=7,max_features="sqrt",max_depth=10)
rf_clf2.fit(processed_x_train, y_train)  # train with tune parameters

In [ ]:
y_pred=rf_clf2.predict(processed_x_test)

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
results['RF/HPT'] = [acc,f1]
results

## Bagging

In [ ]:
from sklearn.ensemble import BaggingClassifier  #import bagging 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

KNN=KNeighborsClassifier()
LR=LogisticRegression()

model_bagg=BaggingClassifier(estimator=KNN,n_estimators=20) ## model objet creation
#base_estimator---> algorithm which you want to pass
#n_estimotors-----> number of base learners

model_bagg.fit(processed_x_train,y_train) ## fitting the model
y_pred=model_bagg.predict(processed_x_test) ## getting the prediction

In [ ]:
acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['Bag/KNN'] = [acc,f1]
results

In [ ]:
model_bagg

In [ ]:
model_bagg=BaggingClassifier(estimator=LR,n_estimators=50) ## model objet creation
#base_estimator---> algorithm which you want to pass
#n_estimotors-----> number of base learners
model_bagg.fit(processed_x_train,y_train) ## fitting the model
y_pred=model_bagg.predict(processed_x_test) ## getting the prediction

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['Bag/LR'] = [acc,f1]
results

In [ ]:
model_bagg=BaggingClassifier(estimator=LR,n_estimators=50,max_samples=0.7,max_features=0.7,bootstrap_features=True) ## model objet creation
#base_estimator---> algorithm which you want to pass
#n_estimotors-----> number of base learners
model_bagg.fit(processed_x_train,y_train) ## fitting the model
y_pred=model_bagg.predict(processed_x_test) ## getting the prediction

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['Bag/LR/Smpl'] = [acc,f1]
results

### Boosting and XG Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier  # Importing GradientBoostingClassifier
gbm=GradientBoostingClassifier() ## object creation
gbm.fit(processed_x_train,y_train) ## fitting the data

y_pred=gbm.predict(processed_x_test)

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
gbm

In [ ]:
results['GB'] = [acc,f1]
results

In [ ]:
## Hyperparameter tuning for GradientBoost
# Importing RandomizedSearchCV from sklearn
from sklearn.model_selection import RandomizedSearchCV

# Define Parameters grid for learning_rate, max_depth, n_estimators
param_grid = {
    'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.4, 0.5, 0.6, 0.7],
    'max_depth': [5, 6, 7, 8, 9, 10],
    'n_estimators': [50, 65, 80, 100]
}
GB=GradientBoostingClassifier()  # Assigning GradientBoostingClassifier model into variables

rcv= RandomizedSearchCV(estimator=GB, scoring='f1',refit = True,param_distributions=param_grid, cv=5,
                               verbose=2, n_jobs=-1)
rcv.fit(processed_x_train,y_train)
rcv.best_params_

In [ ]:
GB2=GradientBoostingClassifier(n_estimators=100, max_depth=7, learning_rate=0.4)
GB2.fit(processed_x_train,y_train)

y_pred=gbm.predict(processed_x_test)

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
results['GB/HPT'] = [acc,f1]
results

## XGBoosting

In [ ]:
#!pip install xgboost #installing model XGBOOST
## model creation
from xgboost import XGBClassifier#importing the model library
xgb_r=XGBClassifier() ## object creation
xgb_r.fit(processed_x_train,y_train)# fitting the data
y_pred=xgb_r.predict(processed_x_test)#predicting the price

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
xgb_r

In [ ]:
results['XGB'] = [acc,f1]
results

In [ ]:
#Hyperparameter tuning
param_grid = {'gamma': [0,0.1,0.2,0.4],
              'learning_rate': [0.01, 0.03, 0.06, 0.1],
              'max_depth': [5,6,7,8,9],
              'n_estimators': [50,65,80],
              'reg_alpha': [0,0.1,0.2,0.4],
              'reg_lambda': [0,0.1,0.2]}

XGB=XGBClassifier(random_state=42,verbosity=0,silent=0)  # Assigning XGBClassifier model into variables
rcv= GridSearchCV(estimator=XGB, scoring='f1',refit=True,param_grid=param_grid,  cv=3,
                               verbose=1, n_jobs=-1)
rcv.fit(processed_x_train,y_train)
rcv.best_params_

In [ ]:
XGB2=XGBClassifier(reg_lambda= 0.1, reg_alpha= 0.1, n_estimators=50, max_depth=8, learning_rate=0.1, gamma=0)
XGB2.fit(processed_x_train,y_train)
y_pred=XGB2.predict(processed_x_test)#testing

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))

In [ ]:
results['XGB/HPT'] = [acc,f1]
results

In [ ]:
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression()
LR.fit(processed_x_train,y_train)

y_pred=LR.predict(processed_x_test)#predicting the price

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))


In [ ]:
## Data Balancing and Creating Model
from imblearn.over_sampling import SMOTE
x_smote,y_smote = SMOTE().fit_resample(processed_x_train,y_train)
LR.fit(x_smote,y_smote)

y_pred=LR.predict(processed_x_test)#predicting the price

acc = accuracy_score(y_test,y_pred)
pr = precision_score(y_test,y_pred)
re = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
cm = confusion_matrix(y_test,y_pred)
print('Accuracy Score: ',acc,'\nPrecision:', pr,'\nRecall Score:', re,'\nF1 Score', f1, '\nConfusion Matrix: \n', cm)
print(classification_report(y_test,y_pred))
